<a href="https://colab.research.google.com/github/hawkeyed-panda/speech_recognition/blob/main/speech.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
yashdogra_speech_commands_path = kagglehub.dataset_download('yashdogra/speech-commands')

print('Data source import complete.')


100%|██████████| 2.25G/2.25G [01:36<00:00, 25.1MB/s]

Extracting files...


Data source import complete.


# Google Speech Commands + HuBERT — Practice 150 phút

**Task:** Keyword Spotting / Audio Classification với 10 lệnh:
`yes, no, up, down, left, right, on, off, stop, go`.

**Dataset:** Kaggle — Speech Commands Dataset v0.02.  
**Model:** `facebook/hubert-base-ls960`.

Mục tiêu:
- waveform, sample rate, duration
- mono, resample 16 kHz, pad/crop
- waveform + spectrogram
- augmentation: noise, time shift
- frozen HuBERT → partial fine-tuning
- Accuracy, Macro-F1, confusion matrix

> Đây là audio classification, nên metric chính là Accuracy/F1; WER/CER dùng cho ASR speech-to-text.

## 0. Setup

In [2]:
import os, gc, random, time
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import torch, torchaudio
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, ConfusionMatrixDisplay
from transformers import AutoFeatureExtractor, HubertForSequenceClassification, get_linear_schedule_with_warmup

SEED=42
TARGET_SR=16000
TARGET_NUM_SAMPLES=16000
BATCH_SIZE=16
NUM_WORKERS=2
VAL_SIZE=0.2
FAST_MODE=False
MAX_PER_CLASS_FAST=500
MODEL_NAME="facebook/hubert-base-ls960"

COMMANDS=["yes","no","up","down","left","right","on","off","stop","go"]
label2id={c:i for i,c in enumerate(COMMANDS)}
id2label={i:c for c,i in label2id.items()}

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:",DEVICE)
if torch.cuda.is_available(): print(torch.cuda.get_device_name(0))

device: cpu


## 1. Dataset

In [3]:
def find_audio_root():
    base=Path("/kaggle/input")
    if not base.exists(): return None
    for root in base.rglob("yes"):
        if root.is_dir():
            parent=root.parent
            if all((parent/x).is_dir() for x in ["yes","no","up","down"]):
                return parent
    return None

AUDIO_ROOT=find_audio_root()
if AUDIO_ROOT is None:
    raise FileNotFoundError("Add Input dataset Speech Commands v0.02 trên Kaggle.")
print("AUDIO_ROOT:",AUDIO_ROOT)

records=[]
for c in COMMANDS:
    files=sorted((AUDIO_ROOT/c).glob("*.wav"))
    if FAST_MODE: files=files[:MAX_PER_CLASS_FAST]
    for f in files:
        records.append({"path":str(f),"label":c,"label_id":label2id[c],
                        "speaker":f.name.split("_nohash_")[0]})
df=pd.DataFrame(records)
print("samples:",len(df))
display(df["label"].value_counts().sort_index().to_frame("count"))

FileNotFoundError: Add Input dataset Speech Commands v0.02 trên Kaggle.

## 2. Nghe một file

In [ ]:
from IPython.display import Audio,display as ipy_display
sample=df.iloc[0]; w,sr=torchaudio.load(sample["path"])
print(sample["label"],w.shape,sr,w.shape[-1]/sr)
ipy_display(Audio(w[0].numpy(),rate=sr))

## 3. Preprocessing — TODO

In [ ]:
def load_audio(path,target_sr=TARGET_SR):
    waveform,sr=torchaudio.load(path)
    # TODO: stereo -> mono
    # TODO: resample -> target_sr
    return waveform.squeeze(0),target_sr

def pad_or_crop(waveform,target_len=TARGET_NUM_SAMPLES):
    # TODO
    return waveform

## 4. Waveform + spectrogram — TODO

In [ ]:
# TODO: plot waveform
# TODO: torchaudio.transforms.Spectrogram(...)

## 5. Augmentation — cho sẵn

In [ ]:
def add_noise(waveform,level=0.003):
    return torch.clamp(waveform+torch.randn_like(waveform)*level,-1,1)
def time_shift(waveform,max_shift=1600):
    return torch.roll(waveform,random.randint(-max_shift,max_shift))

## 6. Split + Dataset/DataLoader — cho sẵn

In [ ]:
train_df,val_df=train_test_split(df,test_size=VAL_SIZE,random_state=SEED,stratify=df["label_id"])
train_df=train_df.reset_index(drop=True); val_df=val_df.reset_index(drop=True)

In [ ]:
class SpeechCommandsDataset(Dataset):
    def __init__(self,frame,augment=False):
        self.df=frame.reset_index(drop=True); self.augment=augment
    def __len__(self): return len(self.df)
    def __getitem__(self,idx):
        r=self.df.iloc[idx]
        w,_=load_audio(r["path"])
        w=pad_or_crop(w)
        if self.augment:
            if random.random()<0.5: w=time_shift(w)
            if random.random()<0.5: w=add_noise(w)
        return {"audio":w.numpy(),"label":int(r["label_id"])}

In [ ]:
feature_extractor=AutoFeatureExtractor.from_pretrained(MODEL_NAME)

def collate_fn(batch):
    audios=[x["audio"] for x in batch]
    labels=torch.tensor([x["label"] for x in batch],dtype=torch.long)
    inputs=feature_extractor(audios,sampling_rate=TARGET_SR,return_tensors="pt",padding=True)
    inputs["labels"]=labels
    return inputs

In [ ]:
train_ds=SpeechCommandsDataset(train_df,augment=True)
val_ds=SpeechCommandsDataset(val_df,augment=False)
train_loader=DataLoader(train_ds,batch_size=BATCH_SIZE,shuffle=True,collate_fn=collate_fn)
val_loader=DataLoader(val_ds,batch_size=BATCH_SIZE,shuffle=False,collate_fn=collate_fn)

## 7. Training utilities — cho sẵn

In [ ]:
def make_model():
    return HubertForSequenceClassification.from_pretrained(
        MODEL_NAME,num_labels=len(COMMANDS),label2id=label2id,id2label=id2label
    )

def count_trainable(model):
    tr=sum(p.numel() for p in model.parameters() if p.requires_grad)
    total=sum(p.numel() for p in model.parameters())
    return tr,total,100*tr/total

@torch.no_grad()
def evaluate(model,loader):
    model.eval(); ls=0; yt=[]; yp=[]
    for batch in loader:
        batch={k:v.to(DEVICE) for k,v in batch.items()}
        out=model(**batch)
        ls+=out.loss.item()*batch["labels"].size(0)
        pred=out.logits.argmax(-1)
        yt.extend(batch["labels"].cpu().numpy()); yp.extend(pred.cpu().numpy())
    yt=np.array(yt); yp=np.array(yp)
    return {"loss":ls/len(loader.dataset),
            "accuracy":accuracy_score(yt,yp),
            "macro_f1":f1_score(yt,yp,average="macro"),
            "y_true":yt,"y_pred":yp}

def fit(model,train_loader,val_loader,optimizer,epochs,tag):
    steps=len(train_loader)*epochs
    scheduler=get_linear_schedule_with_warmup(
        optimizer,int(.1*steps),steps
    )
    best=-1; path=f"/kaggle/working/best_{tag}.pt"; hist=[]
    for ep in range(1,epochs+1):
        model.train(); ls=0; t0=time.time()
        for batch in train_loader:
            batch={k:v.to(DEVICE) for k,v in batch.items()}
            optimizer.zero_grad(set_to_none=True)
            out=model(**batch); loss=out.loss; loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
            optimizer.step(); scheduler.step()
            ls+=loss.item()*batch["labels"].size(0)
        m=evaluate(model,val_loader)
        row={"epoch":ep,"train_loss":ls/len(train_loader.dataset),
             "val_loss":m["loss"],"val_accuracy":m["accuracy"],
             "val_macro_f1":m["macro_f1"],"seconds":time.time()-t0}
        hist.append(row)
        print(f'{tag} {ep}/{epochs} | acc={m["accuracy"]:.4f} | macroF1={m["macro_f1"]:.4f} | {row["seconds"]:.1f}s')
        if m["accuracy"]>best:
            best=m["accuracy"]; torch.save(model.state_dict(),path)
    return pd.DataFrame(hist),path

## 8. Frozen HuBERT — TODO

In [ ]:
# TODO:
# model=make_model().to(DEVICE)
# freeze model.hubert
# train projector + classifier

## 9. Partial fine-tuning — TODO

In [ ]:
# TODO:
# giữ model đã train
# mở model.hubert.encoder.layers[-2:]
# encoder LR nhỏ hơn classifier LR

## 10. Evaluation — TODO

In [ ]:
# TODO: Accuracy, Macro-F1, confusion matrix

## 11. Mini-hackathon
Thử một hypothesis: augmentation / số blocks unfreeze / learning rate.